# Look-ahead · Replication  `[EVAL]`

**What this family answers.** Does the ICLR 2025 look-ahead result (Exp1: Llama-2-7B therapist,
GPT-3.5 patient + oracle, PTO K ∈ {0, 5}, 7 iterations) *replicate* — and which of its claims
survive a change of grader or a change of experiment?

Two parts, both judge-invariant (every table puts the graders side by side, **never averaged**):

1. **Cross-generation (§1).** The *same* Exp1 conversations, re-scored by the Exp3 oracle
   (gpt-4o-mini-2024-07-18, V5 JSON-schema Q1+Q2; `data/eval_scores/_crossgen/`), beside the
   original GPT-3.5 per-conversation scores and ICLR Table 1 as printed. Both the therapist *and*
   the grader changed between Exp1 and Exp3, so the sign reversal (Exp1: K=5 leads; Exp3: K=5
   never leads) is confounded — this isolates the grader. Pairing unit = conversation index
   (= patient permutation; Exp1 did not shuffle personas), n = 96 pairs. No censoring (both arms
   complete 7 iterations).
2. **The ICLR "K=5 is more stable" claim on Exp3 (§2).** Across-persona SD / IQR / ceiling
   shares of Q1, Q2, Q1Q2 per arm × iteration under both graders, Brown-Forsythe and
   Pitman-Morgan variance tests K0 vs K5, and the ceiling-compression check by cooperation level.
   All four Exp3 arms on one axis; pairing unit = `persona_id`; GRPO K=5 right-censored at
   iteration 5.

**Sign convention** everywhere: `delta = K0 − K5`, **+ ⇒ K=0 higher**. vs-Base rows: + ⇒ trained
model higher. Pitman-Morgan `pm_r > 0` ⇒ K=0 *more* dispersed. Exports →
`results/lookahead/replication/{figures,tables}/` (no `<judge>/` level; both graders live inside
every artifact). Numbers ledgers: `crossgen_numbers.json`, `replication_numbers.json`.

The statistics are the promoted paper generators (`eda_analysis.crossgen`,
`eda_analysis.replication`; from `papers/2026_lookahead_pto_grpo/analysis/{crossgen_exp1,
session_shape_stability}.py`) — means / dz / p / counts reproduce the paper's frozen tables
exactly; bootstrap CIs are seeded with the package `BOOT_SEED`, so CI bounds may differ from the
fixture in the third decimal.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 60)

import os, eda_analysis
from eda_analysis import exports, plotting, crossgen, replication
cfg = eda_analysis.EdaConfig(family="lookahead/replication", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # reset_results wiped the banner notebook_setup wrote — re-stamp it

SC = eda_analysis.scores_by_judge(S)     # {'gpt-4o-mini': scores_long, 'claude-haiku-4-5': scores_long}, primary first
JUDGES = list(SC)
print("graders side by side:", JUDGES)

## 1 · Cross-generation — Exp1 (ICLR 2025) under two graders  `[EVAL]`

**Purpose.** Was the ICLR K=5 lead a property of the *transcripts* or of the *grader*? The Exp3
oracle re-scored every Exp1 model state (Base + 7 K=0 + 7 K=5 iterations × 96 conversations); the
original GPT-3.5 per-conversation scores sit beside it. Two graders in every table, never averaged
(they sit on different levels — gpt-4o-mini reads ~0.2–0.4 higher on Final).

### 1a · Load + the two sanity checks
The pairing unit is checked empirically (conversation index `i` = patient permutation `i` in every
model dir), and the transcribed ICLR Table 1 is checked against the on-disk GPT-3.5 means.

In [ ]:
CG   = crossgen.load_crossgen()                       # gpt-4o-mini re-score: model, arm, iteration, conv_index, Q1, Q2, Final
G35  = crossgen.load_exp1_gpt35()                     # the original GPT-3.5 oracle, same shape
ALIGN = crossgen.persona_alignment_check()            # {n_indices, n_consistent, n_conflicting}
T1DEV = crossgen.table1_crosscheck(G35)               # max |ICLR Table 1 - on-disk GPT-3.5 mean|
print("gpt-4o-mini re-score:", CG.shape, "| GPT-3.5 on disk:", G35.shape)
print("persona/index alignment:", ALIGN, "| ICLR Table 1 vs disk max |diff|:", round(T1DEV, 4))
display(crossgen.ICLR_TABLE1.head(4))

### 1b · Levels — every Exp1 model state under both graders  `[EVAL]`
**Purpose.** n / Q1 / Q2 / Final (= mean(Q1, Q2) = the lake's Q1Q2) / SD per model state under
gpt-4o-mini (`*_gpt4omini`), under the original oracle (`*_gpt35`), and ICLR Table 1 as printed
(`*_iclr_tab1`), plus the level gap between graders. Rows: Base, K=0 iters 1–7, K=5 iters 1–7.

In [ ]:
LV = crossgen.levels(CG, G35)
display(LV.round(3))
exports.save_table(LV, "crossgen_levels", caption=crossgen.CAPTIONS["levels"])

### 1c · The look-ahead contrast at matched iteration, and its summaries  `[EVAL]`
**Purpose.** `mean_delta = K0 − K5` per iteration under each grader (**+ ⇒ K=0 higher**), paired on
conversation index (n = 96): dz, 95% bootstrap CI, Wilcoxon *p*, Holm within (grader, metric)
across the 7 iterations, paired-*t*, the sign split, and the unpaired Welch reading. GPT-3.5's
deltas are heavy-tailed, so read its sign split next to *p*. The summary table adds the ICLR
best-vs-best pick (L0_I4 vs L5_I7), each grader's own best-vs-best, the pooled-arm contrast and
the "every K=5 model above every K=0 model" ordering claim.

In [ ]:
KC = crossgen.k_contrast(CG, G35)
display(KC[["grader", "metric", "iteration", "n", "mean_delta", "dz", "ci_lo", "ci_hi", "p", "p_holm",
            "n_K0_higher", "n_K5_higher"]].round(4))
exports.save_table(KC, "crossgen_kcontrast", caption=crossgen.CAPTIONS["kcontrast"])

SUMM_CG = crossgen.k_summary(CG, G35)
ORD = crossgen.ordering_claims(CG, G35)
display(SUMM_CG.round(4)); display(ORD)
exports.save_table(SUMM_CG, "crossgen_kcontrast_summary", caption=crossgen.CAPTIONS["kcontrast_summary"])

### 1d · Grader agreement on Exp1's conversations  `[EVAL]`
**Purpose.** Do the two graders *rank* the 15 model states the same way even though their levels
differ? Spearman ρ (and Pearson *r*) at the level of the 15 model means, the 14 trained-model
means (so Base does not anchor the rank), and per conversation (15 × 96 pooled).

In [ ]:
AGR = crossgen.grader_agreement(LV, CG, G35)
display(AGR.round(4))
exports.save_table(AGR, "crossgen_grader_agreement", caption=crossgen.CAPTIONS["grader_agreement"])

### 1e · Each trained Exp1 model vs the untrained Base  `[EVAL]`
**Purpose.** Did training help at all under each grader? `mean_delta = model − Base`
(**+ ⇒ trained model higher**), paired on conversation index, Holm within (grader, arm, metric)
across the 7 iterations.

In [ ]:
VB = crossgen.vs_base(CG, G35)
display(VB[VB.metric == "Final"][["grader", "arm", "iteration", "n", "mean_delta", "dz", "p_holm",
                                   "n_model_higher", "n_base_higher"]].round(4))
exports.save_table(VB, "crossgen_vsbase", caption=crossgen.CAPTIONS["vsbase"])

### 1f · The K=3 sweep on disk (GPT-3.5 only)  `[EVAL]`
**Purpose.** Exp1's `LookAhead_3` (4 iterations) is the only look-ahead *dose* data on disk, but it
ran at different hyper-parameters (therapist temperature 0.7 / filter τ 0.2 vs 0.9 / 0.1) and the
re-scoring tool deliberately excludes it. Reported as its ORIGINAL GPT-3.5 means, with a
no-API-call estimate of what re-scoring it would cost. Not comparable to §1b without that caveat.

In [ ]:
LA3 = crossgen.la3_gpt35()
LA3_COST = crossgen.la3_cost_estimate()
display(LA3.round(3)); print("re-score cost estimate (no API call made):", LA3_COST)
exports.save_table(LA3, "crossgen_la3_gpt35", caption=crossgen.CAPTIONS["la3_gpt35"])

### 1g · Figure — Exp1 by iteration under two graders  `[EVAL]`
Final = mean(Q1, Q2) by PTO iteration, K=0 solid circles vs K=5 dashed squares, Base dotted, bands =
95% bootstrap CI of the mean; left = the original GPT-3.5 oracle, right = the same transcripts under
gpt-4o-mini. Separate y-axes per panel (different grader levels; never averaged). `crossgen_col` is
the same figure stacked for a single column.

In [ ]:
fig = plotting.crossgen.crossgen_fig(G35, CG, layout="wide")
exports.save_fig(fig, "crossgen", caption=crossgen.CAPTIONS["fig"])
plt.show()
fig_c = plotting.crossgen.crossgen_fig(G35, CG, layout="col")
exports.save_fig(fig_c, "crossgen_col", caption="Single-column layout of `crossgen` (same data, panels stacked): "
                 + crossgen.CAPTIONS["fig"])
plt.show()

### 1h · Numbers ledger  `[EVAL]`
Every quotable number of §1 as `crossgen_numbers.json` (`{dotted.key: {value, source, note}}`), keys
mirroring the paper ledger `analysis/out/crossgen_exp1.json`; `source` strings point at the tables
saved above (`tables/crossgen_<name>.md`).

In [ ]:
F = dict(crossgen=CG, gpt35=G35, alignment=ALIGN, table1_max_abs_diff=T1DEV, levels=LV, kcontrast=KC,
         summary=SUMM_CG, ordering=ORD, agreement=AGR, vsbase=VB, la3=LA3, la3_cost=LA3_COST)
NUM_CG = crossgen.crossgen_numbers(F, table_prefix="tables/crossgen_")
NUM_CG["fig.caption"] = {"value": crossgen.CAPTIONS["fig"], "source": "figures/crossgen.png", "note": ""}
exports.save_numbers("crossgen_numbers", NUM_CG,
                     caption="Number ledger for the cross-generation link (Exp1 under two graders): pairing/crosscheck, "
                             "levels per model state, K contrast per iteration and its summaries, grader agreement, "
                             "vs-Base, the K=3 status, and the per-grader verdict. Sign: delta = K0 - K5, + => K=0 higher; "
                             "pairing on conversation index; no censoring.")
print(len(NUM_CG), "ledger keys; verdicts:")
for g in (crossgen.GRADER_GPT4OMINI, crossgen.GRADER_GPT35):
    v = NUM_CG[f"verdict.{g}.Final"]["value"]
    print(f"  {g:12s} K=5 higher at {v['n_iters_K5_higher']}/7 iters "
          f"({v['n_iters_K5_higher_p_holm_lt_05']} Holm-sig), mean of 7 deltas {v['mean_of_7_deltas']:+.3f}, median dz {v['median_dz']:+.3f}")

## 2 · The ICLR "K=5 is more stable" claim on Exp3  `[EVAL]`

**Purpose.** The ICLR poster read K=5's lower across-conversation SD as stability. Re-tested on all
four Exp3 arms under both graders: SD / IQR / ceiling shares per arm × iteration (§2a), the
variance contrast K0 vs K5 per matched iteration with two tests (§2b) and its tally, where the
lowest SD sits and how SD tracks the mean (§2c), and the ceiling-compression check by cooperation
level (§2d). On a bounded 1–5 scale a higher mean mechanically compresses SD, so a low SD next to a
high ceiling share is scale compression, not stability. Pairing unit = `persona_id` (the
per-iteration file shuffle replayed; never `file_index`); iteration 0 = two independent base draws
(a free noise-floor row); GRPO_LA5 right-censored at iteration 5 (matched iterations 0..5 for GRPO,
0..10 for PTO).

### 2a · Dispersion per arm × iteration, both graders  `[EVAL]`

In [ ]:
SD = replication.sd_by_iter(SC)                       # judge, metric, arm, method, K, iteration, n, mean, median, sd, iqr, ceiling shares
display(SD[SD.metric == "Q1Q2"].pivot(index=["judge", "arm"], columns="iteration", values="sd").round(3))
exports.save_table(SD, "sd_by_iter", caption=replication.CAPTIONS["sd"])

### 2b · Variance contrast K0 vs K5 per matched iteration + tally  `[EVAL]`
**Purpose.** `sd_ratio_K5_over_K0 < 1` ⇒ the K=5 arm is *less* dispersed. Brown-Forsythe treats the
two arms as independent groups; Pitman-Morgan is the persona-paired variance test (`pm_r > 0` ⇒
K=0 more dispersed). Holm within each (judge, method, metric) family across iterations. The tally
counts, over the trained matched iterations (PTO 1..10, GRPO 1..5), how often K=5 has the lower SD /
IQR and how many contrasts are Holm-significant in each direction; `iter0_sd_*` are the two base
draws (the noise floor for an SD difference).

In [ ]:
BF = replication.sd_tests(SC)
display(BF[["judge", "method", "metric", "iteration", "n", "sd_K0", "sd_K5", "sd_ratio_K5_over_K0",
            "bf_p_holm", "pm_r", "pm_p_holm"]].round(4))
exports.save_table(BF, "sd_tests", caption=replication.CAPTIONS["sd_bf"])

TALLY = replication.sd_tally(BF)
display(TALLY.round(3))
exports.save_table(TALLY, "sd_tally", caption=replication.CAPTIONS["sd_tally"])

### 2c · Where the lowest SD sits, and does SD just track the mean?  `[EVAL]`
**Purpose.** Per grader × rubric over the trained states (iteration 0 excluded): the state with the
smallest across-persona SD and its mean, the state with the highest mean and its SD, Spearman ρ
between mean and SD across all trained states (strongly negative = dispersion tracks the ceiling,
not the optimizer), and each arm's own lowest SD + where it occurs.

In [ ]:
SUMM_SD = replication.sd_summary(SD)
display(SUMM_SD[["judge", "metric", "min_sd_arm", "min_sd_iteration", "min_sd", "min_sd_mean",
                 "max_mean_arm", "max_mean_iteration", "max_mean", "max_mean_sd",
                 "spearman_mean_vs_sd", "spearman_p", "n_states"]].round(3))
exports.save_table(SUMM_SD, "sd_summary", caption=replication.CAPTIONS["sd_summary"])

### 2d · Ceiling compression by cooperation level  `[EVAL]`
**Purpose.** Share of conversations at Q1Q2 ≥ 4.5 / == 5 per arm × iteration and per patient
cooperation level (Resistant / WarmsUp / Cooperative, 32 personas each), both graders. The held-out
judge never awards ≥ 4.5 on Q1Q2 (its max is 4.25), so its ceiling shares are 0 by construction —
its SD is the cleaner read of dispersion.

In [ ]:
CEIL = replication.ceiling(SC)
display(CEIL[["judge", "arm", "iteration", "mean_all", "sd_all", "share_ge45_all", "share_eq5_all",
              "share_ge45_Resistant", "share_ge45_WarmsUp", "share_ge45_Cooperative"]].round(3))
exports.save_table(CEIL, "ceiling", caption=replication.CAPTIONS["ceiling"])

### 2e · Figure — SD of the training reward by iteration under each grader  `[EVAL]`
Across-persona SD of Q1Q2 by iteration, one panel per grader (never averaged; the ring marks that
grader's lowest-SD trained state), plus SD vs mean over the trained states (filled = gpt-4o-mini,
open = claude-haiku-4-5). K=0 solid filled, K=5 dashed open; PTO cool / GRPO warm; GRPO_LA5 stops
at iteration 5.

In [ ]:
# sd_fig is laid out for a 7.2-inch, default-font page (the paper's rc); under the EDA-wide seaborn
# 'notebook' context its right-panel x-label overflows the tight bbox, so render THIS figure under
# the 'paper' context (fonts ~0.8x) — same data, same statistics, the paper's look.
import seaborn as sns
with sns.plotting_context("paper"):
    fig = plotting.replication.sd_fig(SD, SUMM_SD, metric="Q1Q2", judges=JUDGES, palette=S.PALETTE)
    exports.save_fig(fig, "sd", caption=replication.CAPTIONS["fig_sd"] + " Ring = that grader's lowest-SD trained state "
                     "(sd_summary). K=0 solid filled, K=5 dashed open; PTO cool / GRPO warm; iteration 0 = the two "
                     "independent base draws; " + replication.CENSOR)
plt.show()

### 2f · Numbers ledger  `[EVAL]`
Every quotable number of §2 as `replication_numbers.json`, keys mirroring the paper ledger
`analysis/out/session_shape_stability.json` (`sd.*`, `sd_bf.*`, `sd_tally.*`, `sd_summary.*`,
`ceiling.*`). The shape / length / selection keys of that ledger belong to `lookahead/behaviour`.
`source` strings are rewritten to the table names saved here (`sd_by_iter.md`, `sd_tests.md`).

In [ ]:
NUM_RP = replication.replication_numbers(sd=SD, bf=BF, tally=TALLY, summary=SUMM_SD, ceil=CEIL, table_prefix="tables/")
_RENAME = {"tables/sd.md": "tables/sd_by_iter.md", "tables/sd_bf.md": "tables/sd_tests.md"}
for k, rec in NUM_RP.items():
    for old, new in _RENAME.items():
        if rec.get("source", "").startswith(old):
            rec["source"] = new + rec["source"][len(old):]
exports.save_numbers("replication_numbers", NUM_RP,
                     caption="Number ledger for the ICLR-stability replication on Exp3: per arm x iteration dispersion "
                             "(sd.*), the K0-vs-K5 variance tests (sd_bf.*), their tally (sd_tally.*), the lowest-SD summary "
                             "(sd_summary.*) and the ceiling check (ceiling.*), both graders (never averaged). Sign: + => K=0 "
                             "higher / more dispersed; pairing on persona_id; GRPO_LA5 censored at iteration 5.")
print(len(NUM_RP), "ledger keys")
display(TALLY[["judge", "method", "metric", "n_iters", "n_K5_lower_sd", "median_sd_ratio_K5_over_K0",
               "n_pm_holm_sig_K5_lower", "n_pm_holm_sig_K0_lower"]])

## 3 · Index
Prune caption lines whose artifact was not regenerated and rebuild `results/lookahead/INDEX.md` +
`results/INDEX.md`.

In [ ]:
exports.prune_orphan_captions(); exports.build_index()